## Website Scrape & Ingestion

Crawl `rc.virginia.edu` + `learning.rc.virginia.edu` + `archive.rc.virginia.edu` and comvert into markdown files for knowledge base

Pipeline consistent with jira-cloud-full / md-new / video-v2:
- Metadata: only `source_type`, `source`, `chunk_number` (no `tags`/`date_updated`)

## 2. Crawl Websites

Recursively crawl all three domains. Extracts text from `<article>` elements, strips images and metadata tags.

In [2]:
import cloudscraper
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse, urljoin
import time

SKIP_EXTENSIONS = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".svg", ".pdf", ".zip", ".tar", ".gz", ".mp4", ".webp")
ALLOWED_NETLOCS = {"rc.virginia.edu", "learning.rc.virginia.edu", "archive.rc.virginia.edu"}

scraper = cloudscraper.create_scraper()

def is_valid(url, allowed_netlocs=ALLOWED_NETLOCS):
    parsed = urlparse(url)
    path = parsed.path.lower()
    return (
        parsed.scheme in {"http", "https"}
        and parsed.netloc in allowed_netlocs
        and not path.endswith(SKIP_EXTENSIONS)
    )

def crawl(url, visited=set(), documents={}, netloc=None):
    if url in visited:
        return documents
    visited.add(url)

    netloc = netloc or urlparse(url).netloc

    try:
        response = scraper.get(url, timeout=15)

        if response.url != url:
            url = response.url
            if url in visited:
                return documents
            visited.add(url)

        content_type = response.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            return documents
        if response.status_code != 200:
            return documents

        soup = BeautifulSoup(response.text, "html.parser")

        if len(articles := soup.find_all("article")) != 1:
            print(f"Skipping {url} (no single article)")
        else:
            article_soup = BeautifulSoup(str(articles[0]), "html.parser")

            for tag in article_soup.find_all("img"):
                tag.decompose()

            return_link = article_soup.find("a", string=re.compile(r"^\u00ab Return to"))
            if return_link:
                return_link.decompose()

            metadata_tag = article_soup.find("p", class_="blog-post-meta")

            for a_tag in article_soup.find_all("a", href=True):
                href = a_tag["href"]
                if not href.startswith("http"):
                    href = urljoin(url, href)
                a_tag["href"] = href
                a_tag.string = f"[{a_tag.get_text(strip=True)}]({href})"

            title_tag = article_soup.find("h2", class_="blog-post-title")
            if title_tag:
                title_text = title_tag.get_text(strip=True)
                title_tag.string = f"# {title_text}\n\n"

            for h1_tag in article_soup.find_all("h1"):
                h1_text = h1_tag.get_text(strip=True)
                h1_tag.string = f"\n\n## {h1_text}\n"

            if metadata_tag:
                metadata_tag.decompose()

            text = article_soup.get_text()
            text = re.sub(r"https?:\/\/\S+?\.png", "", text)
            text = re.sub(r"\S+\.png", "", text)
            text = re.sub(r"\n{3,}", "\n\n", text)
            text = re.sub(r"[ \t]+", " ", text)

            documents[url] = {
                "text": text.strip(),
                "source": url,
            }
            print(f"Extracted: {url}")

        for a_tag in soup.find_all("a", href=True):
            next_url = a_tag["href"]
            if not next_url.startswith(("http://", "https://")):
                next_url = urljoin(url, next_url)
            next_url = next_url.split("#")[0]

            if next_url not in visited and is_valid(next_url):
                crawl(next_url, visited, documents, netloc)

        time.sleep(0.2)

    except Exception as e:
        print(f"Failed to crawl {url}: {e}")

    return documents

In [3]:
documents = crawl("https://rc.virginia.edu/")
documents.update(crawl("https://learning.rc.virginia.edu/"))
print(f"\nTotal pages crawled: {len(documents)}")

Extracted: https://rc.virginia.edu/
Skipping https://rc.virginia.edu/cacsdrupal/login (no single article)
Skipping https://rc.virginia.edu/system-status (no single article)
Skipping https://rc.virginia.edu/news (no single article)
Extracted: https://rc.virginia.edu/about
Extracted: https://rc.virginia.edu/contact-0
Extracted: https://rc.virginia.edu/getting-started
Extracted: https://rc.virginia.edu/services
Extracted: https://rc.virginia.edu/request-manage
Extracted: https://rc.virginia.edu/training
Extracted: https://rc.virginia.edu/support
Extracted: https://rc.virginia.edu/i-want-to
Extracted: https://rc.virginia.edu/getting-started/complete-orientation
Extracted: https://rc.virginia.edu/services/pricing
Extracted: https://rc.virginia.edu/services/data-analytics-center/support-grants-and-proposals
Extracted: https://rc.virginia.edu/system-status/rivanna-afton
Extracted: https://rc.virginia.edu/system-status/ivy-rio
Extracted: https://rc.virginia.edu/system-status/storage
Extracted:

## 3. Sitemap Gap Fill

Fetch URLs from sitemaps that the link-following crawler missed.

In [4]:
import xml.etree.ElementTree as ET

SKIP_PATTERNS = ['/author/', '/category/', '/tag/']

def get_sitemap_urls(sitemap_url, skip_patterns=None):
    skip_patterns = skip_patterns or []
    ns = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
    resp = scraper.get(sitemap_url, timeout=15)
    root = ET.fromstring(resp.text)
    base = 'https://' + sitemap_url.split('/')[2]
    raw = [url.find('sm:loc', ns).text for url in root.findall('sm:url', ns)]
    urls = [base + u if u.startswith('/') else u for u in raw]
    return [u for u in urls if not any(p in u for p in skip_patterns)]

def extract_article(url):
    response = scraper.get(url, timeout=15)
    if response.status_code != 200 or 'text/html' not in response.headers.get('Content-Type', ''):
        return None
    soup = BeautifulSoup(response.text, 'html.parser')
    articles = soup.find_all('article')
    if len(articles) != 1:
        return None
    article_soup = BeautifulSoup(str(articles[0]), 'html.parser')
    for tag in article_soup.find_all('img'):
        tag.decompose()
    metadata_tag = article_soup.find('p', class_='blog-post-meta')
    for a_tag in article_soup.find_all('a', href=True):
        href = a_tag['href']
        if not href.startswith('http'):
            href = urljoin(url, href)
        a_tag['href'] = href
        a_tag.string = '[' + a_tag.get_text(strip=True) + '](' + href + ')'
    if metadata_tag:
        metadata_tag.decompose()
    text = article_soup.get_text()
    text = re.sub(r'https?:\/\/\S+?\.png', '', text)
    text = re.sub(r'\S+\.png', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return {'text': text.strip(), 'source': url}

rc_urls    = get_sitemap_urls('https://rc.virginia.edu/sitemap.xml')
learn_urls = get_sitemap_urls('https://learning.rc.virginia.edu/sitemap.xml', SKIP_PATTERNS)
all_sitemap_urls = rc_urls + learn_urls

missing = [u for u in all_sitemap_urls if u not in documents]
print(f"Sitemap total: {len(all_sitemap_urls)}, already crawled: {len(all_sitemap_urls)-len(missing)}, missing: {len(missing)}")

for url in missing:
    try:
        result = extract_article(url)
        if result:
            documents[url] = result
            print(f"Added: {url}")
        else:
            print(f"Skipped (no article): {url}")
        time.sleep(0.2)
    except Exception as e:
        print(f"Failed {url}: {e}")

print(f"\nTotal documents after gap fill: {len(documents)}")

Sitemap total: 715, already crawled: 588, missing: 127
Added: https://rc.virginia.edu/events/introduction-uva-rc-genai
Added: https://rc.virginia.edu/events/office-hours-every-thursday-1
Added: https://rc.virginia.edu/index
Skipped (no article): https://rc.virginia.edu/system-status
Added: https://rc.virginia.edu/examples-landing-page
Added: https://rc.virginia.edu/menu-work
Added: https://rc.virginia.edu/test-landing-page
Skipped (no article): https://rc.virginia.edu/people/alex-ptak
Skipped (no article): https://rc.virginia.edu/people/jacalyn-huband-phd
Skipped (no article): https://rc.virginia.edu/people/mark-rucker-phd
Skipped (no article): https://rc.virginia.edu/people/joshua-baller-phd
Skipped (no article): https://rc.virginia.edu/people/michael-e-navicky
Skipped (no article): https://rc.virginia.edu/people/madeline-bornstad
Skipped (no article): https://rc.virginia.edu/people/gladys-andino-phd
Skipped (no article): https://rc.virginia.edu/people/deb-triant-phd
Skipped (no artic

## 3b. Patch JS-rendered Pages

Some pages (e.g. the Slurm Script Generator) are interactive JavaScript tools that the static scraper cannot render, resulting in empty or title-only content. Manually supply descriptive text for these pages so they can be retrieved by the RAG pipeline.

In [5]:
manual_patches = {
    "https://rc.virginia.edu/userinfo/hpc/slurm-script-generator/": {
        "text": """# Slurm Script Generator

The UVA Research Computing Slurm Script Generator is an interactive web tool that helps users create Slurm job submission scripts for the UVA HPC system.

## How to Use

Visit the Slurm Script Generator at: https://archive.rc.virginia.edu/userinfo/hpc/slurm-script-generator/

The tool allows you to:
- Select a partition (queue) for your job (e.g., standard, gpu, parallel, dev)
- Specify the number of nodes and cores (tasks) needed
- Set memory requirements per core or per node
- Configure wall time (how long the job will run)
- Set up GPU resources if needed (number and type of GPUs)
- Specify your allocation group
- Add email notifications for job start, end, or failure
- Generate a ready-to-use Slurm batch script (.slurm file)

## When to Use This Tool

Use the Slurm Script Generator if you are:
- New to HPC and need help writing your first Slurm script
- Unsure about which Slurm directives (#SBATCH) to include
- Looking for a quick way to generate a template job script
- Wanting to explore what options are available for different partitions

## Example Output

The generator produces a script like:
```
#!/bin/bash
#SBATCH --job-name=myjob
#SBATCH --partition=standard
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=02:00:00
#SBATCH -A mygroup

module load your_software
your_command_here
```

## Related Resources

- Slurm Job Manager documentation: https://rc.virginia.edu/userinfo/hpc/slurm/
- HPC Getting Started: https://rc.virginia.edu/getting-started
- Sample Slurm scripts by partition: https://rc.virginia.edu/userinfo/hpc/slurm/
""",
        "source": "https://rc.virginia.edu/userinfo/hpc/slurm-script-generator/",
    },
    "https://rc.virginia.edu/userinfo/hpc/software/physics/": {
        "text": """# Physics Software on UVA HPC

UVA Research Computing provides several physics simulation and computational physics software packages on the HPC system.

## Available Physics Software

The following physics-related software is available via the module system on UVA HPC:

- **LAMMPS** - Large-scale Atomic/Molecular Massively Parallel Simulator for molecular dynamics
- **GROMACS** - Molecular dynamics package for simulating proteins, lipids, and nucleic acids
- **Quantum ESPRESSO** - Integrated suite for electronic-structure calculations and materials modeling
- **VASP** - Vienna Ab initio Simulation Package for atomic scale materials modelling
- **OpenFOAM** - Computational fluid dynamics (CFD) toolbox
- **NAMD** - Parallel molecular dynamics for large biomolecular systems

## How to Use

Load physics software using the module system:
```
module load lammps
module load gromacs
module load quantumespresso
```

Use `module spider <software_name>` to see available versions and dependencies.

## Related Resources

- Full software list: https://rc.virginia.edu/userinfo/hpc/software/
- Slurm job submission: https://rc.virginia.edu/userinfo/hpc/slurm/
""",
        "source": "https://rc.virginia.edu/userinfo/hpc/software/physics/",
    },
}

patched = 0
for url, doc in manual_patches.items():
    if url not in documents or len(documents.get(url, {}).get("text", "")) < 100:
        documents[url] = doc
        patched += 1
        print(f"Patched: {url}")
    else:
        print(f"Already has content: {url}")

print(f"\nManually patched {patched} JS-rendered pages.")
print(f"Total documents: {len(documents)}")

Patched: https://rc.virginia.edu/userinfo/hpc/slurm-script-generator/
Patched: https://rc.virginia.edu/userinfo/hpc/software/physics/

Manually patched 2 JS-rendered pages.
Total documents: 1092


## 5. Preview

In [9]:
for doc in website_documents[:3]:
    print(f"=== {doc.metadata['source']} ===")
    print(doc.page_content[:300])
    print()

=== https://rc.virginia.edu/ ===
News and Events

[ImageSmall Data Analytics Resource Award Proposals Due June 29](https://rc.virginia.edu/news/small-data-analytics-resource-award-proposals-due-june-29)

[ImageMajor AI and GPU Expansion at UVA](https://rc.virginia.edu/news/major-ai-and-gpu-expansion-uva)

[ImageWelcome to Our New W

=== https://rc.virginia.edu/about ===
Image
 

 
A Message From Our AVP
 

Since 2023, I have had the privilege of leading UVA Research Computing (UVA-RC). In that time, the UVA-RC team has focused on building out a new array of services that address the needs of both our classical high-performance computing community and the many areas

=== https://rc.virginia.edu/contact-0 ===
How to Reach UsOpen a Support Ticket[Submit a Support Request](https://forms.rc.virginia.edu/form/support-request/) Request a Consultation We offer consultations for many of our services and types of support.[Learn more about ways to work with us](https://rc.virginia.edu/services/ex